In [ ]:
import pandas as pd
import io
import zipfile
from google.colab import files
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

print("Vui lòng bấm 'Choose Files' để tải file dataset.zip lên:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as z:
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
    with z.open(csv_filename) as f:
        df = pd.read_csv(f)

# 1. Làm sạch dữ liệu (Xử lý các giá trị 0 thành NaN rồi điền Median)
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)
for col in cols_to_fix:
    df[col] = df[col].fillna(df[col].median())

X = df.drop(columns=['Outcome', 'index'] if 'index' in df.columns else ['Outcome'])
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Xây dựng Pipeline và huấn luyện đồng thời 4 mô hình
# Mỗi model đều được bọc Pipeline gồm StandardScaler riêng để chống Data Leakage tuyệt đối
models_dict = {
    'logistic': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'svm': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(probability=True, random_state=42))
    ]),
    'naive_bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', GaussianNB())
    ]),
    'random_forest': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

print("--- ĐANG HUẤN LUYỆN 4 MÔ HÌNH ---")
for name, pipeline_model in models_dict.items():
    pipeline_model.fit(X_train, y_train)
    acc = pipeline_model.score(X_test, y_test)
    print(f"Model [{name}] - Test Accuracy: {acc:.4f}")

# 3. Đóng gói dictionary chứa 4 mô hình thành 1 file model.joblib duy nhất
joblib.dump(models_dict, 'model.joblib', compress=3)
print("\n Đã xuất thành công file 'model.joblib' chứa đủ 4 mô hình cho hệ thống!")

# Tải file về máy để đưa vào thư mục ai-models/models/
files.download('model.joblib')